# 🧠📍 Smart Tagging

Metrica's **automatic** tactical annotation of the tactical video: phases of play and set
pieces, each one a time interval with a code and a set of tags.

This notebook covers loading it, what is in it, and the two things that will bite you if you
treat it as ground truth.

---

## Loading

The file is a **SportsCode XML**, the standard format in football performance analysis, so
[kloppy](https://kloppy.pysport.org/user-guide/loading-data/sportscode/) reads it directly.

`games.asset_paths()` builds the path, so nothing here spells a filename out and
`MTAH_DATA_DIR` keeps working if you store the bundle elsewhere.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from kloppy import sportscode

import games

path = games.asset_paths()['smart_tagging']
events = sportscode.load(path).to_df()
print('%d instances' % len(events))
events.head()

## What is in it

Each row is one tagged interval:

- **timestamp / end_timestamp** — start and end, as pandas timedeltas from the first frame
- **code** — the phase or set piece, upper case (`SET PIECES`, `BUILD UP`, …)
- **Team** — which side the phase belongs to
- **Half** — `1st Half` or `2nd Half`, as strings
- plus per-code tags: `Type`, `Side`, `Max Players in the box`, `Direction of ball entry`

Note the capital in `Team`: kloppy passes the tag groups through with the names the XML uses, so
every tag column is title case. The full vocabulary is in [DATA.md](../DATA.md).

In [ ]:
events['code'].value_counts()

### Codes are upper case, tag values are title case

`SET PIECES` and `Kick Off (Start)`. Match case-insensitively unless you enjoy silent empty
results.

In [ ]:
set_pieces = events[events['code'].str.lower() == 'set pieces']
set_pieces['Type'].value_counts()

## Converting to video frames

The timestamps are **seconds from the first frame of the tactical video**, so a video frame —
and therefore a submission row — is just `round(seconds * 25)`. There is no offset to discover.

`starter/load_smart_tagging.py` does this without kloppy, and re-checks the time base against
both kickoffs every time it runs.

In [ ]:
FPS = games.GAMES[games.GAME]['fps']

frames = pd.DataFrame({
    'code': events['code'],
    'Team': events['Team'],
    'start_frame': (events['timestamp'].dt.total_seconds() * FPS).round().astype(int),
    'end_frame': (events['end_timestamp'].dt.total_seconds() * FPS).round().astype(int),
})
frames.head()

## Two things to be careful about

### 1. `Team` is not always a team

36 instances are tagged `N/A` — including **every throw-in**, which are exactly the restarts
where knowing the side would be worth the most. Handle it explicitly; do not let it fall
through to a default.

In [ ]:
events['Team'].value_counts(dropna=False)

### 2. An interval is a window, not an instant

A set piece is tagged as a span of a few seconds *around* the restart, not at the frame the ball
is struck. The two `Kick Off (Start)` tags below both **contain** the real kickoff rather than
starting on it — they open about 3.5 seconds early.

So `start_frame` is not the restart frame, and the same imprecision applies to every other
interval in the file. Use a tag to say *roughly when* something happened, and the tracking to say
exactly when.

In [ ]:
kickoffs = set_pieces[set_pieces['Type'].str.lower() == 'kick off (start)'].copy()
kickoffs['start_frame'] = (kickoffs['timestamp'].dt.total_seconds() * FPS).round().astype(int)
kickoffs['end_frame'] = (kickoffs['end_timestamp'].dt.total_seconds() * FPS).round().astype(int)
kickoffs['real_kickoff'] = [11185, 113742]          # from the ground truth
kickoffs['contains_it'] = ((kickoffs['start_frame'] <= kickoffs['real_kickoff'])
                           & (kickoffs['real_kickoff'] <= kickoffs['end_frame']))

kickoffs[['Half', 'start_frame', 'end_frame', 'real_kickoff', 'contains_it']]

---

## Why this is worth using anyway

Every phase carries a **team** tag, and every possession has a team. Every set piece is a
restart, and every restart ends a dead-ball stretch. The correspondence is not exact — that is
the challenge — but it is a strong prior, for free, on both halves of the label you are asked
to produce.

Just remember the intervals are *phases*, not possessions: a Build Up can span one possession,
stop short of one, or cover two.

**Next:** [`02_tracking_atd.ipynb`](02_tracking_atd.ipynb) for the tracking data, then
[`03_first_submission.ipynb`](03_first_submission.ipynb) to build and score a submission.